# QAOA Tutorial for Max-Cut

This notebook is organized into the following four main steps:

1. Start with a Max-Cut graph example.
2. Construct the corresponding Hamiltonian and QAOA circuit.
3. Solve the QAOA optimization problem.
4. Visualize the results.

After that, we repeat the same experiment with a noise model derived from a fake IBM backend to compare the noiseless and noisy cases.

## Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from qaoa_maxcut import (
    assignment_to_partition,
    build_graph,
    build_fake_backend_simulator,
    build_maxcut_hamiltonian,
    build_qaoa_circuit,
    FakeBackendConfig,
    cut_value,
    draw_graph,
    optimal_solution_probability,
    plot_sampling_distribution,
    rank_assignments,
    solve_maxcut_instance,
)

plt.style.use('seaborn-v0_8-whitegrid')

## 1. Start with a Max-Cut Example

We begin by explicitly specifying a graph. All Hamiltonians, circuits, and optimization steps below are built for this graph.

In [ ]:
example_edges = [
    (0, 1),
    (0, 2),
    (1, 2),
    (1, 3),
    (2, 4),
    (3, 4),
]

graph = build_graph(example_edges)
print('Nodes:', list(graph.nodes()))
print('Edges:', list(graph.edges()))

draw_graph(graph, title='Max-Cut graph example');

## 2. Construct the Hamiltonian and QAOA Circuit

For Max-Cut, a standard cost Hamiltonian is:

$$
H_C = \sum_{(i,j) \in E} \frac{1}{2}(I - Z_i Z_j)
$$

Each edge contributes one term. If the two endpoints are assigned to different partitions, that edge contributes 1 to the cut. QAOA alternates between:

- the cost unitary generated by $H_C$
- the mixer unitary, typically based on $H_M = \sum_i X_i$


In [ ]:
hamiltonian = build_maxcut_hamiltonian(graph)
print('Max-Cut Hamiltonian:')
print(hamiltonian)

In [ ]:
layers = 2
circuit, gamma, beta = build_qaoa_circuit(graph, layers)
print('gamma parameters:', list(gamma))
print('beta parameters:', list(beta))
display(circuit.draw('mpl', fold=-1))

The following bitstring example illustrates how a cut is encoded. For example, `00110` means:

- nodes with bit `0` are placed on one side
- nodes with bit `1` are placed on the other side

In [ ]:
example_assignment = '00110'
left, right = assignment_to_partition(example_assignment)
print('assignment:', example_assignment)
print('partition:', left, '|', right)
print('cut value:', cut_value(example_assignment, graph))

draw_graph(graph, assignment=example_assignment, title=f'Example cut {example_assignment}');

## 3. Solve QAOA

In this step we do two things:

- use a classical optimizer to search for the QAOA parameters `gamma` and `beta`
- compute the exact optimum by brute force for comparison

In [ ]:
solution = solve_maxcut_instance(
    edges=example_edges,
    layers=layers,
    shots=2048,
    restarts=8,
    seed=7,
)

result = solution.result
exact_cut = solution.exact_cut
exact_assignments = solution.exact_assignments
optimal_prob = optimal_solution_probability(result.counts, graph, exact_assignments)

print('Expected cut value from QAOA: {:.4f}'.format(result.expected_cut))
print('gamma =', np.round(result.x[:layers], 6).tolist())
print('beta  =', np.round(result.x[layers:], 6).tolist())
print('Exact optimum cut value:', exact_cut)
print('Optimal assignments:', exact_assignments)
print('Probability of sampling an optimal solution: {:.4%}'.format(optimal_prob))

## 4. Visualize the Results

We present the noiseless results from three angles:

- the objective value during optimization
- the sampling distribution over all possible bitstrings in computational-basis order
- a visualization of the best sampled cut

In [ ]:
history = [-value for value in result.objective_history]
plt.figure(figsize=(7, 4))
plt.plot(history, linewidth=2)
plt.xlabel('Objective evaluation')
plt.ylabel('Estimated cut value')
plt.title('Noiseless QAOA optimization trace');

In [ ]:
plot_sampling_distribution(result.counts, graph, title='Noiseless sampling distribution');

In [ ]:
ranked = rank_assignments(result.cut_distribution, graph, limit=5)
for assignment, frequency, value in ranked:
    left, right = assignment_to_partition(assignment)
    print(f'assignment={assignment}  shots={frequency:4d}  cut={value}  partition={left} | {right}')

In [ ]:
best_assignment = ranked[0][0]
best_cut = cut_value(best_assignment, graph)
draw_graph(graph, assignment=best_assignment, title=f'Best sampled cut: {best_assignment} (cut={best_cut})');

In [ ]:
print('Best exact cut value:', exact_cut)
print('One optimal assignment:', exact_assignments[0])
print('Best sampled assignment:', best_assignment)
print('Best sampled cut value:', best_cut)
print('Approximation ratio:', best_cut / exact_cut)
print('Probability of sampling an optimal solution: {:.4%}'.format(optimal_prob))

## Additional Experiment: Repeat with a Fake IBM Backend Noise Model

The previous experiment used a noiseless simulator. Now we rerun the same graph with a noise model derived from a fake IBM backend.

This gives us a more device-like noisy model, including basis gates, coupling map, and backend-derived error data.

This section requires both `qiskit-aer` and `qiskit-ibm-runtime`.

In [ ]:
fake_backend_config = FakeBackendConfig(backend_name='FakeManilaV2')
fake_backend, noisy_backend = build_fake_backend_simulator(fake_backend_config)
print(fake_backend_config)
print(fake_backend)

In [ ]:
noisy_solution = solve_maxcut_instance(
    edges=example_edges,
    layers=layers,
    shots=2048,
    restarts=8,
    seed=7,
    simulator_backend=noisy_backend,
)

noisy_result = noisy_solution.result
noisy_optimal_prob = optimal_solution_probability(noisy_result.counts, graph, exact_assignments)
print('Noisy expected cut value: {:.4f}'.format(noisy_result.expected_cut))
print('Noisy gamma =', np.round(noisy_result.x[:layers], 6).tolist())
print('Noisy beta  =', np.round(noisy_result.x[layers:], 6).tolist())
print('Probability of sampling an optimal solution under noise: {:.4%}'.format(noisy_optimal_prob))

In [ ]:
fig_width = max(8, 0.45 * (2 ** graph.number_of_nodes()))
fig, axes = plt.subplots(2, 1, figsize=(fig_width, 8), constrained_layout=True)
plot_sampling_distribution(result.counts, graph, ax=axes[0], title='Noiseless sampling distribution')
plot_sampling_distribution(noisy_result.counts, graph, ax=axes[1], title='Noisy sampling distribution')

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot([-value for value in result.objective_history], label='Noiseless', linewidth=2)
plt.plot([-value for value in noisy_result.objective_history], label='Noisy', linewidth=2)
plt.xlabel('Objective evaluation')
plt.ylabel('Estimated cut value')
plt.title('Noiseless vs noisy optimization traces')
plt.legend();

In [ ]:
noisy_ranked = rank_assignments(noisy_result.cut_distribution, graph, limit=5)
for assignment, frequency, value in noisy_ranked:
    left, right = assignment_to_partition(assignment)
    print(f'assignment={assignment}  shots={frequency:4d}  cut={value}  partition={left} | {right}')

In [ ]:
noisy_best_assignment = noisy_ranked[0][0]
noisy_best_cut = cut_value(noisy_best_assignment, graph)
draw_graph(graph, assignment=noisy_best_assignment, title=f'Best noisy sampled cut: {noisy_best_assignment} (cut={noisy_best_cut})');

In [ ]:
print('Exact optimum cut value:', exact_cut)
print('Noiseless expected cut value: {:.4f}'.format(result.expected_cut))
print('Noisy expected cut value: {:.4f}'.format(noisy_result.expected_cut))
print('Best noiseless sampled cut:', best_cut)
print('Best noisy sampled cut:', noisy_best_cut)
print('Noiseless probability of sampling an optimal solution: {:.4%}'.format(optimal_prob))
print('Noisy probability of sampling an optimal solution: {:.4%}'.format(noisy_optimal_prob))